# 🚇 서울 지하철 혼잡도 & 사고 분석 (1~9호선) — 정책입안자 대상
**SCQA · 민토 피라미드 · 기초통계 · 시각화 → 제안 + Action Plan**

| SCQA | 내용 |
|---|---|
| **S 상황** | 코로나 후 수요 회복(2024년 91%)으로 출퇴근 혼잡 재심화 |
| **C 문제** | 이미 혼잡한 노선·역·시간대에 **사고·급정차가 겹치면 혼잡 폭증(안전 위험)**. 5년 사고 2,837건 분석 → 08시 최다·출퇴근 34.4% 집중 |
| **Q 질문** | 어느 노선·역·시간대가 **"사고 시 가장 위험한 취약 지점"**인가? |
| **A 답** | 2호선·도심 업무지구·출퇴근에 혼잡·사고 동시 집중 → **취약 지점 우선 투자 + 사고·수요 대응** |

> 범위: 서울 1~9호선 / "혼잡"=역 승하차 인원 / 단면=최근월·추이=전체

## 0. 환경 설정 & 데이터 로드

In [ ]:
# (Colab 최초 1회) 한글 폰트
!sudo apt-get install -y -qq fonts-nanum >/dev/null 2>&1
!fc-cache -fv >/dev/null 2>&1
!rm -rf ~/.cache/matplotlib

In [ ]:
import pandas as pd, numpy as np, re, os
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# 한글 폰트 자동 선택 (Colab: NanumBarunGothic / mac: AppleGothic / Win: Malgun Gothic)
_avail = {f.name for f in fm.fontManager.ttflist}
for _f in ["NanumBarunGothic", "AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if _f in _avail:
        plt.rcParams["font.family"] = _f; break
plt.rcParams["axes.unicode_minus"] = False

# 발표 덱과 동일한 절제 팔레트 + Wilke 시각화 기본값
NAVY, STEEL, GRAYD, GRAYL, RED, BG = "#112E51", "#205493", "#6D7882", "#C7CDD4", "#C0392B", "#F4F5F6"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlecolor": NAVY,
    "axes.labelsize": 11, "axes.labelcolor": "#1E2124",
    "xtick.labelsize": 10, "ytick.labelsize": 10, "xtick.color": GRAYD, "ytick.color": GRAYD,
    "axes.edgecolor": GRAYD, "axes.linewidth": 0.8, "axes.grid": False,
})
def clean(ax, keep=("bottom", "left")):
    """Wilke식 군더더기 제거: 불필요한 축선·눈금 제거"""
    for sp in ("top", "right", "bottom", "left"):
        ax.spines[sp].set_visible(sp in keep)
    ax.tick_params(length=0)
    return ax

# 데이터 파일 자동 탐색 (로컬/Colab 어디에 올리든 OK)
def D(fn):
    import glob
    for q in [fn, "data/" + fn, "../" + fn, "/content/" + fn, "/content/data/" + fn]:
        if os.path.exists(q): return q
    h = glob.glob(f"**/{fn}", recursive=True)
    return h[0] if h else fn

print("환경 설정 완료 · 폰트:", plt.rcParams["font.family"])


In [ ]:
# 승하차 데이터 (★ cp949)
df = pd.read_csv(D('서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv'), encoding='cp949')
print('원본:', df.shape)

## 1. 데이터 정제 (총승하차 + 중복제거 + 1~9호선)

In [ ]:
승차컬=[c for c in df.columns if c.endswith('승차인원')]
하차컬=[c for c in df.columns if c.endswith('하차인원')]
df['총승차']=df[승차컬].sum(axis=1); df['총하차']=df[하차컬].sum(axis=1)
df['총승하차']=df['총승차']+df['총하차']
df=df.drop_duplicates(subset=['사용월','호선명','지하철역'])
df['연도']=df['사용월']//100
df=df[df['호선명'].str.match(r'^[1-9]호선')].copy()
df['호선']=df['호선명'].str.extract(r'^([1-9]호선)')
LATEST=df['사용월'].max(); cur=df[df['사용월']==LATEST].copy()
print('정제 후:', df.shape, '| 기준월:', LATEST, f"({cur['지하철역'].nunique()}역)")

## 2. 근거 1 — 연도별 추이 (코로나 충격/회복)

In [ ]:
# 근거1 — 연도별 총 이용객 (2026 부분연도 제외)
yr = df[df["연도"] <= 2025].groupby("연도")["총승하차"].sum() / 1e8
print(yr.round(2).to_dict())
print(f"코로나 2020: {(yr[2020]/yr[2019]-1)*100:+.1f}%  |  2024 회복: {yr[2024]/yr[2019]*100:.0f}%")

fig, ax = plt.subplots()
ax.axvspan(2019.7, 2020.5, color=GRAYL, alpha=.55, zorder=0)
ax.text(2020.1, yr.min()*1.01, "코로나", color=GRAYD, fontsize=9, ha="center")
ax.plot(yr.index, yr.values, marker="o", color=NAVY, lw=2.6, ms=6, zorder=3)
ax.annotate(f"{yr.iloc[-1]:.1f}억", (yr.index[-1], yr.iloc[-1]),
            textcoords="offset points", xytext=(8, 2), color=NAVY, fontweight="bold", fontsize=11)
clean(ax); ax.set_title("근거1 · 연도별 총 이용객 — 코로나 급감 후 빠른 재혼잡")
ax.set_ylabel("억 명"); ax.set_ylim(0, yr.max()*1.15)
ax.grid(axis="y", color=GRAYL, alpha=.5); plt.tight_layout(); plt.show()


## 3. 근거 2 — 호선별 차이 (ANOVA)

In [ ]:
# 근거2 — 호선별 차이 (ANOVA + 효과크기 + Welch + Tukey + t-test)
line = cur.groupby("호선")["총승하차"].agg(총합="sum", 역당평균="mean", 역수="count").sort_values("총합", ascending=False)
print(line.round(0))

groups = [x["총승하차"].values for _, x in cur.groupby("호선") if len(x) >= 5]
F, p = stats.f_oneway(*groups)
allv = np.concatenate(groups); gm = allv.mean()
ssb = sum(len(g)*(g.mean()-gm)**2 for g in groups); sst = ((allv-gm)**2).sum(); eta2 = ssb/sst
W, pw = stats.levene(*groups)
print(f"\n[ANOVA]  F={F:.1f}, p={p:.2e},  η²={eta2:.3f} ({'대' if eta2>=.14 else '중'} 효과)")
print(f"[Levene] p={pw:.2e} → 등분산 {'위배 → Welch 권장' if pw<.05 else 'OK'}")

# t-test: 2호선 vs 그 외 (Welch + Cohen's d)
l2 = cur[cur['호선']=='2호선']['총승하차']; ot = cur[cur['호선']!='2호선']['총승하차']
t, pt = stats.ttest_ind(l2, ot, equal_var=False)
nx, ny = len(l2), len(ot); sp = np.sqrt(((nx-1)*l2.std(ddof=1)**2+(ny-1)*ot.std(ddof=1)**2)/(nx+ny-2))
d = (l2.mean()-ot.mean())/sp
print(f"[t-test] 2호선 vs 그외 {l2.mean()/ot.mean():.2f}배,  Welch t={t:.2f}, p={pt:.2e}, Cohen's d={d:.2f} (큰 효과)")
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    tuk = pairwise_tukeyhsd(cur['총승하차'], cur['호선'])
    sig2 = sum(1 for r in tuk.summary().data[1:] if '2호선' in (r[0], r[1]) and r[-1])
    print(f"[Tukey] 2호선이 다른 {sig2}개 호선과 유의차")
except Exception:
    print("[Tukey] statsmodels 미설치 → 생략")

# 시각화: 값순 정렬 + 2호선만 강조 + 직접 라벨 (Wilke)
v = (line["총합"]/1e6).sort_values()
fig, ax = plt.subplots()
ax.barh(v.index, v.values, color=[RED if i=="2호선" else GRAYL for i in v.index])
for i, (nm, val) in enumerate(v.items()):
    ax.text(val+max(v)*0.01, i, f"{val:.0f}", va="center", fontsize=9,
            color=RED if nm=="2호선" else GRAYD, fontweight="bold" if nm=="2호선" else "normal")
clean(ax, keep=("bottom",)); ax.set_title("근거2 · 호선별 총 이용객 — 2호선 압도 (t·ANOVA 유의, η²=.28)")
ax.set_xlabel("백만 명"); ax.set_xlim(0, max(v)*1.12); ax.set_yticks(range(len(v))); plt.tight_layout(); plt.show()


## 3-2. 호선 유형화 (MECE) — 혼잡 원인은 호선마다 다르다

**단일 기준 = '지배적 병목 위치'**(수요÷공급)로 1~9호선을 분류 → 유형이 겹치지 않고(ME) 9개 전부 포함(CE).
- **A. 구조적 고혼잡 = 공급 병목** (2·9): 수요는 큰데 편성·배차 한계 → 공급 확대
- **B. 시간·방향 쏠림 = 수요 병목** (3·4·6·7·8): 주거–업무 분리 쌍봉·편방향 → 수요 분산
- **C. 광역환승·노후 = 인프라 병목** (1·5): 환승 부하+노후 → 인프라 개량

9호선은 역당 이용 최하위인데 혼잡 최상위(급행 195%) = 수요예측 실패·6량 공급부족.
(출처: 서울교통공사 혼잡도 · 서울연구원 보고서 · KCI 「9호선 혼잡도 개선 비용효과분석」)


In [ ]:
# 호선 유형화 (MECE): 단일 기준 '지배적 병목'(수요÷공급)으로 1~9호선 분류
ams=["07시-08시 승차인원","08시-09시 승차인원"]; amh=["07시-08시 하차인원","08시-09시 하차인원"]
rows=[]
for ln,g in cur.groupby("호선"):
    amS=g[ams].sum().sum(); amH=g[amh].sum().sum()
    rows.append({"호선":ln,"역수":len(g),"역당평균(만)":round(g["총승하차"].mean()/1e4),
                 "아침유입비":round(amH/amS,2),"최혼잡역":g.nlargest(1,"총승하차")["지하철역"].iloc[0]})
line=pd.DataFrame(rows).sort_values("역당평균(만)",ascending=False).reset_index(drop=True)
typ={"2호선":"A.구조적 고혼잡","9호선":"A.구조적 고혼잡",
     "3호선":"B.시간·방향 쏠림","4호선":"B.시간·방향 쏠림","6호선":"B.시간·방향 쏠림","7호선":"B.시간·방향 쏠림","8호선":"B.시간·방향 쏠림",
     "1호선":"C.광역환승·노후","5호선":"C.광역환승·노후"}
병목={"A.구조적 고혼잡":"공급","B.시간·방향 쏠림":"수요","C.광역환승·노후":"인프라"}
line["유형"]=line["호선"].map(typ); line["병목"]=line["유형"].map(병목)
print(line[["호선","역수","역당평균(만)","아침유입비","유형","병목"]].to_string(index=False))

# --- MECE 검증 (ME: 호선당 유형 1개·중복 0 / CE: 1~9호선 전부·누락 0) ---
assert line["호선"].notna().all() and not line["유형"].isna().any(), "ME 위반: 미분류 호선 존재"
assert len(line)==9 and set(line["호선"])==set(f"{i}호선" for i in range(1,10)), "CE 위반: 9개 호선 누락"
print("\n[MECE 검증 통과] 단일 기준(지배적 병목)으로 분류 → ME(중복 0)·CE(9개 전부, 누락 0)")
print("  공급 병목 A: 2·9   |   수요 병목 B: 3·4·6·7·8   |   인프라 병목 C: 1·5")
print("[처방] A 공급확대(증결·증차·CBTC) · B 수요분산(시차·비피크 할인) · C 인프라개량(노후·환승)")

# 시각화: 역당 이용 막대(유형색) + 9호선 역설 주석
col={"A.구조적 고혼잡":RED,"B.시간·방향 쏠림":STEEL,"C.광역환승·노후":GRAYD}
ls=line.sort_values("역당평균(만)")
fig,ax=plt.subplots(figsize=(9,4.6))
ax.barh(ls["호선"],ls["역당평균(만)"],color=[col[t] for t in ls["유형"]])
for i,(_,r) in enumerate(ls.iterrows()):
    ax.text(r["역당평균(만)"]+2,i,f'{int(r["역당평균(만)"])}만',va="center",fontsize=9,color=GRAYD)
yi=list(ls["호선"]).index("9호선")
ax.annotate("9호선: 역당 이용 최하위인데\n혼잡 최상위(급행 195%) → 공급 부족",
            xy=(ls.iloc[yi]["역당평균(만)"],yi), xytext=(95,yi+1.5),
            fontsize=9.5,color=RED,fontweight="bold",arrowprops=dict(arrowstyle="->",color=RED,lw=1.3))
clean(ax,keep=("bottom",))
ax.set_title("근거2-2 · 호선 유형화(MECE) — 역당 이용객 & 병목 유형  (A:레드 B:스틸 C:회색)")
ax.set_xlabel("역당 평균 이용객(만 명)"); plt.tight_layout(); plt.show()


## 4. 근거 3 — 시간대 출퇴근 피크 & 유입/유출형

In [ ]:
# 근거3 — 시간대별 출퇴근 쌍봉 피크 (유입/유출형)
승t = cur[승차컬].sum(); 하t = cur[하차컬].sum()
hours = [c.split("시-")[0] for c in 승차컬]
print("승차 피크:", 승t.idxmax(), "| 하차 피크:", 하t.idxmax())
am승 = ["07시-08시 승차인원", "08시-09시 승차인원"]; am하 = ["07시-08시 하차인원", "08시-09시 하차인원"]
cur["유입비"] = cur[am하].sum(axis=1)/(cur[am승].sum(axis=1)+1)
big = cur[cur[am승].sum(axis=1) > 5000]
print("유입형(업무지구):", list(big.nlargest(5, "유입비")["지하철역"]))
print("유출형(주거지):", list(big.nsmallest(5, "유입비")["지하철역"]))

x = list(range(len(승차컬)))
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.axvspan(3, 5, color=GRAYL, alpha=.5, zorder=0)    # 출근(07~09)
ax.axvspan(14, 16, color=GRAYL, alpha=.5, zorder=0)  # 퇴근(18~20)
ax.plot(x, 하t.values/1e6, "-", color=RED, lw=2.4, zorder=3)
ax.plot(x, 승t.values/1e6, "-", color=STEEL, lw=2.4, zorder=3)
i_am, i_pm = 4, 14
ax.annotate("하차(출근 유입)", (i_am, 하t.values[i_am]/1e6), textcoords="offset points",
            xytext=(6, 6), color=RED, fontweight="bold", fontsize=10)
ax.annotate("승차(퇴근 유출)", (i_pm, 승t.values[i_pm]/1e6), textcoords="offset points",
            xytext=(6, 6), color=STEEL, fontweight="bold", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(hours, rotation=90, fontsize=8)
clean(ax); ax.set_title("근거3 · 시간대별 승하차 — 출근 08시·퇴근 18시 쌍봉 피크")
ax.set_ylabel("백만 명"); ax.grid(axis="y", color=GRAYL, alpha=.5); plt.tight_layout(); plt.show()


## 5. 근거 4 — 역별 혼잡 Top & 집중도 (+ 지도)

In [ ]:
# 근거4 — 역별 혼잡 Top & 집중도 (소수 핫스팟 집중)
top = cur.nlargest(10, "총승하차")[["지하철역", "호선", "총승하차"]]
print(top.to_string(index=False))
s = cur["총승하차"].sort_values(ascending=False).values; n = max(1, len(s)//10)
conc = s[:n].sum()/s.sum()*100
print(f"\n상위 10% ({n}역) = 전체 이용객의 {conc:.1f}% 집중")

t10 = top.iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(t10["지하철역"], t10["총승하차"]/1e6, color=[RED if h=="2호선" else GRAYL for h in t10["호선"]])
for i, val in enumerate(t10["총승하차"]/1e6):
    ax.text(val+max(t10["총승하차"]/1e6)*0.01, i, f"{val:.0f}", va="center", fontsize=9, color=GRAYD)
clean(ax, keep=("bottom",)); ax.set_title(f"근거4 · 최혼잡 역 Top10 — 상위10%가 전체의 {conc:.0f}%")
ax.set_xlabel("백만 명"); plt.tight_layout(); plt.show()
# 실제 지도(folium): charts/지하철_혼잡_지도.html → 발표 덱엔 charts/근거4_folium지도.png 로 임베드


## 6. 근거 5 — 사고도 출퇴근·혼잡역에 몰린다 ⭐
서울교통공사 5년 사고 2,837건(발생시간 포함) 직접 분석 → 뉴스가 아닌 데이터로 입증.

In [ ]:
# 근거5 — 사고도 출퇴근·혼잡 시간에 겹친다 (+ 노출 보정·포아송 회귀)
acc = pd.read_csv(D("서울교통공사_사고현황_5년.csv"), encoding="cp949")
acc["시"] = acc["발생시간"].astype(str).str.extract(r"^(\d{1,2}):").astype(float)
acc = acc.dropna(subset=["시"]); acc["시"] = acc["시"].astype(int); N = len(acc)
hc = acc["시"].value_counts().sort_index()
peak = acc["시"].isin([7, 8, 18, 19]).sum()
print(f"총 {N}건 | 최다 {hc.idxmax()}시 {hc.max()}건 | 출퇴근(7·8·18·19시) {peak/N*100:.1f}% (균등이면 ~21%)")
print("사고 다발역:", list(acc["발생역"].value_counts().head(5).index))

# --- 노출(이용객) 보정: 시간대별 '백만 명당 사고율' (cur 이용객 기준) ---
ridh = {h: (cur.get(f"{h:02d}시-{h+1:02d}시 승차인원", pd.Series([0])).sum()
            + cur.get(f"{h:02d}시-{h+1:02d}시 하차인원", pd.Series([0])).sum()) for h in range(4, 24)}
ex = pd.DataFrame({"시": list(range(4, 24))})
ex["사고"] = ex["시"].map(hc).fillna(0); ex["이용객"] = ex["시"].map(ridh)
ex = ex[ex["이용객"] > 0].copy(); ex["백만명당"] = ex["사고"] / ex["이용객"] * 1e6
r8 = ex[ex["시"] == 8].iloc[0]
rush = ex[ex["시"].isin([7, 8, 9])]; off = ex[ex["시"].isin([11, 12, 13, 14, 15, 16])]
print(f"\n[노출 보정] 08시: 건수 x{r8['사고']/ex['사고'].mean():.2f} / 이용객 x{r8['이용객']/ex['이용객'].mean():.2f} / 백만명당 사고율 x{r8['백만명당']/ex['백만명당'].mean():.2f}")
print(f"           출근(7-9) vs 낮비피크(11-16) 사고율비 x{(rush['사고'].sum()/rush['이용객'].sum())/(off['사고'].sum()/off['이용객'].sum()):.2f} (≈ 비슷)")
try:
    import statsmodels.api as sm
    ex2 = ex.copy(); ex2["출근"] = ex2["시"].isin([7, 8, 9]).astype(int)
    m = sm.GLM(ex2["사고"], sm.add_constant(ex2["출근"]), family=sm.families.Poisson(), offset=np.log(ex2["이용객"])).fit()
    print(f"[포아송 회귀] 출근 사고율 = 비출근의 {np.exp(m.params['출근']):.2f}배 (offset=log 이용객)")
except Exception:
    print("[포아송] statsmodels 미설치 → 생략")
print("→ '건수 2.65배'는 이용객(노출) 효과가 큼. 보정 시 1.1~1.7배 → 1인당 발생률 차이는 작다.")
print("→ 핵심: '빈도'가 아니라 '사고가 출근 피크 밀집(≈4명/㎡)에 겹치는 영향'")

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(hc.index, hc.values, color=[RED if h in (7, 8, 18, 19) else GRAYL for h in hc.index])
ax.text(8, hc.get(8, 0), f"{hc.get(8,0)}", ha="center", va="bottom", color=RED, fontweight="bold")
clean(ax, keep=("bottom",)); ax.set_title(f"근거5 · 시간대별 지하철 사고 ({N}건) — 출퇴근(적색) 집중")
ax.set_xlabel("시"); ax.set_ylabel("건수"); ax.grid(axis="y", color=GRAYL, alpha=.5); ax.set_xticks(range(0, 24, 2)); plt.tight_layout(); plt.show()


## 8. 근거 6 — 역 유형 군집 (K-means, ISLP) → 유형별 맞춤 대책

In [ ]:
# 근거6 — K-means 군집: 역을 '업무지구·주거·상시형'으로 자동 분류
am승 = ["07시-08시 승차인원", "08시-09시 승차인원"]; am하 = ["07시-08시 하차인원", "08시-09시 하차인원"]
f = cur.copy()
f["AM유입비"] = f[am하].sum(axis=1)/(f[am승].sum(axis=1)+1)   # 아침 하차/승차 → 업무지구일수록↑
f["규모"] = np.log1p(f["총승하차"])
f = f[f["총승하차"] > f["총승하차"].quantile(0.05)].copy()      # 하위5% 이상치 제거(군집 안정)
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    X = StandardScaler().fit_transform(f[["AM유입비", "규모"]].fillna(0))
    km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
    f["군집"] = km.labels_
    prof = f.groupby("군집").agg(역수=("지하철역", "count"), AM유입비=("AM유입비", "mean"), 평균이용=("총승하차", "mean"))
    order = prof.sort_values("AM유입비").index.tolist()
    names = {order[0]: "주거형(유출)", order[1]: "혼합·상시형", order[2]: "업무지구형(유입)"}
    f["유형"] = f["군집"].map(names)
    print(f.groupby("유형").agg(역수=("지하철역", "count"), 평균AM유입비=("AM유입비", "mean")).round(2))
    cmap = {"주거형(유출)": STEEL, "혼합·상시형": GRAYD, "업무지구형(유입)": RED}
    fig, ax = plt.subplots(figsize=(9, 4.6))
    for typ, g in f.groupby("유형"):
        ax.scatter(g["AM유입비"], g["총승하차"]/1e6, s=26, alpha=.75, color=cmap[typ], label=f"{typ} ({len(g)}역)")
    clean(ax); ax.set_title("근거6 · 역 유형 군집(K-means) — 업무지구·주거·상시형 3유형")
    ax.set_xlabel("아침 유입비 (하차/승차) →  업무지구일수록 큼"); ax.set_ylabel("총 이용객(백만)")
    ax.set_xscale("log"); ax.legend(frameon=False, fontsize=9, loc="upper right")
    ax.grid(alpha=.3, color=GRAYL); plt.tight_layout(); plt.show()
except Exception as e:
    print("[K-means] scikit-learn 미설치 → 생략:", e)


## 7. 🏛️ 핵심 결론 → 제안 → Action Plan (정책입안자 / 투트랙)

> **핵심 결론**: 서울 지하철은 **2호선·도심 업무지구·출퇴근(08·18시)**에 혼잡이 구조적으로 집중되고, **사고도 같은 시간대·혼잡 환승역에 몰린다**(2,837건 중 출퇴근 34.4%, 08시 최다). 코로나 후 재혼잡 + 사고 증가(+81%)로 위험이 커지는 중.

**근거**: ①2020 –27%→재혼잡 ②2호선 압도(ANOVA p<0.001) ③출근 하차 08시·퇴근 승차 18시 ④상위10% 역=28.5% ⑤**사고 08시 최다·출퇴근 34.4%·다발역=혼잡역**

| 트랙 | 제안 | Action Plan |
|---|---|---|
| 🚇 서울교통공사 | 집중역 우선 관리 + **사고 비상 혼잡 대응** | 단기 혼잡·사고 실시간 안내·인력 / 상시 신호·전기 설비 예방정비 / 중기 2호선·집중역 동선 개선 |
| 🏛️ 국회 국토위 | 수요 분산 + **노후 설비 예산** | 시차출퇴근 인센티브 / 노후 신호·전동차 교체 예산 / 안전기준 법제화 |

*범위: 서울 1~9호선 / 혼잡=승하차 / 사고=5년 2,837건*
